In [1]:
import cv2
import time

from scr import tracker, detection, catboost_filter, feature_generation, tools

OpenCV: Couldn't read video stream from file "all_video/video_name.mp4"


In [2]:
'''
Ячейка дубль файла config – в рабочей версии планируется запуск через main_ball.py.

После этой ячейки код адаптирован для работы в формате ноутбука
'''
#MODELS_PARAMS
BALL_MODEL_PATH = "/Users/pasha/Projects/Data_stat/model/ball.onnx"
CAT_BOOST_MODEL_PATH = "/Users/pasha/Projects/Data_stat/model/catboost_ball_model.cbm"

INPUT_W = 512
INPUT_H = 288
HEATMAP_TRASH = 0.5
SEQ_LEN = 9

Probability = 0.8

#VIDEO_PARAMS
INPUT_VIDEO_PATH = "/Users/pasha/Projects/Data_stat/git/all_video/short_test.mp4"
OUTPUT_VIDEO_PATH = "/Users/pasha/Projects/Data_stat/git/all_video/result_traks_and_videos/short_test_with_detect.mp4"

zero_cap = cv2.VideoCapture(INPUT_VIDEO_PATH)

FPS = zero_cap.get(cv2.CAP_PROP_FPS)
VIDEO_WIDTH = int(zero_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VIDEO_HEIGHT = int(zero_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

#TRACKER_PARAMS
GAP_WITH_CONFIRMED = 30
GAP_WITHOUT_CONFIRMED = 5
LEN_MAX_TRACK = 15

In [3]:
first_cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
total_frames = int(first_cap.get(cv2.CAP_PROP_FRAME_COUNT))

BALL_MODEL = detection.load_model(BALL_MODEL_PATH)
CAT_BOOST_FILTER = catboost_filter.cat_boost_load(CAT_BOOST_MODEL_PATH)

input_name, output_name, _, _, _ = detection.model_params(BALL_MODEL)

frame_id_counter = 0
track_id_counter = 0

2026-08-02 20:08:15.734 Python[55306:1590551] 2026-08-02 20:08:15.734707 [W:onnxruntime:, coreml_execution_provider.cc:137 GetCapability] CoreMLExecutionProvider::GetCapability, number of partitions supported by CoreML: 5 number of nodes in the graph: 236 number of nodes supported by CoreML: 89


In [4]:
buffer_frame = []
all_track = {}
database = {}

In [5]:
start_time = time.time()

while first_cap.isOpened():

    ret, frame = first_cap.read()
    if not ret:
        break

    if frame_id_counter % 30 == 0 or frame_id_counter == total_frames:
        tools.progress_bar_drawing(frame_id_counter, total_frames, start_time)

    frame_id_counter += 1
    preprocessed_frame = detection.preprocess_for_onnx(frame, INPUT_W, INPUT_H)
    buffer_frame.append(preprocessed_frame)

    if len(buffer_frame) == SEQ_LEN:

        #Detection_part

        INPUT = detection.input_tensor(buffer_frame)
        buffer_frame.pop(0)

        OUTPUT = detection.inference(BALL_MODEL, input_name, output_name, INPUT)
        visible, decode_x, decode_y = detection.heatmap_decoder(OUTPUT, HEATMAP_TRASH)

        x, y = detection.original_dimension(decode_x, decode_y,
                                            INPUT_W, INPUT_H,
                                            VIDEO_WIDTH, VIDEO_HEIGHT)

        detect = detection.create_dict(frame_id_counter, visible, x, y)

        #Tracker_part

        matching = tracker.match_detection(all_track, detect)

        if matching:
            matching_id = list(matching.keys())[-1]
            tracker.update_track(all_track[matching_id], detect, False)

        elif detect["visible"] == 1:

            if len(all_track) > LEN_MAX_TRACK:

                all_track = tracker.clearing_track_queue(all_track, LEN_MAX_TRACK)

            track_id_counter += 1
            all_track[track_id_counter] = tracker.create_track(detect)

        all_track = tracker.gap_cleaner(all_track, detect,
                                        GAP_WITH_CONFIRMED, GAP_WITHOUT_CONFIRMED)

        main_id = tracker.select_main_ball(all_track)

        if main_id is not None:

            catboost_track = feature_generation.feature_generator(all_track[main_id])

            if len(all_track[main_id]["x"]) >= 2:

                is_a_ball = catboost_filter.cat_boost_filter(
                    catboost_track, CAT_BOOST_FILTER, Probability)

            else:
                is_a_ball = False

            if is_a_ball:

                all_track[main_id]["confirmed"] = True
                database[main_id] = all_track[main_id]

[██████████████████████████████] 99% | На обработку 2.93 минут видео ушло 7.37 минут

In [6]:
light_databese = {}

for track_id in database.keys():
    track = database[track_id]

    for i in range(len(database[track_id]["frame_idx"])):

        light_databese[database[track_id]["frame_idx"][i]] = (database[track_id]["x"][i], database[track_id]["y"][i]), int(track_id)

keys_list = list(light_databese.keys())

In [ ]:
total_frames = int(first_cap.get(cv2.CAP_PROP_FRAME_COUNT))

In [9]:
second_cap = cv2.VideoCapture(INPUT_VIDEO_PATH)

save_format = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, save_format, FPS, (VIDEO_WIDTH, VIDEO_HEIGHT))

frame_id_counter = 0

while second_cap.isOpened():
    ret, frame = second_cap.read()

    if not ret:
        break

    frame_id_counter +=1
    print(f"Обработка {frame_id_counter}/{total_frames}")
    if frame_id_counter in keys_list:

        frame = cv2.circle(frame, (light_databese[frame_id_counter][0]), 15, (0, 255, 0), 3)
        frame = cv2.putText(frame, str(light_databese[frame_id_counter][1]),(10, 30),cv2.FONT_HERSHEY_SIMPLEX,1,(255, 255, 255),2)

    out.write(frame)

second_cap.release()
out.release()
cv2.destroyAllWindows()

Обработка 1/5280
Обработка 2/5280
Обработка 3/5280
Обработка 4/5280
Обработка 5/5280
Обработка 6/5280
Обработка 7/5280
Обработка 8/5280
Обработка 9/5280
Обработка 10/5280
Обработка 11/5280
Обработка 12/5280
Обработка 13/5280
Обработка 14/5280
Обработка 15/5280
Обработка 16/5280
Обработка 17/5280
Обработка 18/5280
Обработка 19/5280
Обработка 20/5280
Обработка 21/5280
Обработка 22/5280
Обработка 23/5280
Обработка 24/5280
Обработка 25/5280
Обработка 26/5280
Обработка 27/5280
Обработка 28/5280
Обработка 29/5280
Обработка 30/5280
Обработка 31/5280
Обработка 32/5280
Обработка 33/5280
Обработка 34/5280
Обработка 35/5280
Обработка 36/5280
Обработка 37/5280
Обработка 38/5280
Обработка 39/5280
Обработка 40/5280
Обработка 41/5280
Обработка 42/5280
Обработка 43/5280
Обработка 44/5280
Обработка 45/5280
Обработка 46/5280
Обработка 47/5280
Обработка 48/5280
Обработка 49/5280
Обработка 50/5280
Обработка 51/5280
Обработка 52/5280
Обработка 53/5280
Обработка 54/5280
Обработка 55/5280
Обработка 56/5280
О